1. `mkdir 5_Lab_Streamlit`
2. `cd 5_Lab_Streamlit`
3. `uv venv`
4. `source .venv/bin/activate`
5. `uv init`
6. `!uv add streamlit`
7. `uv run streamlit run main.py`

`mkdir resources && touch api_altair.py api_kpis.py data_loader.py gapm_seaborn.py main.py `

### main.py

```python
import streamlit as st

from api_altair import plot_altair_benchmark, plot_altair_insights
from api_kpis import plot_kpis
from data_loader import filter_df_by_year, get_data_api, load_csv_data
from gapm_seaborn import plot_life_exp_dist, plot_seaborn_swarm

st.set_page_config(layout="wide", page_title="Dashboard Pro", page_icon="📊")

# --- SIDEBAR: CONFIGURACIÓN ---
st.sidebar.title("🛠️ Configuración")

api_url_input = st.sidebar.text_input(
    "URL",
    value="https://restcountries.com/v3.1/all?fields=name,region,population,area,capital,cca3",
)

df_api = get_data_api(api_url_input)

region = st.sidebar.selectbox("Región", sorted(df_api["region"].unique()))
df_api_filtered = df_api[df_api["region"] == region]

st.sidebar.divider()

uploaded_file = st.sidebar.file_uploader("Gapminder CSV", type=["csv"])

# 1. Cargamos la data (con cache)
df_raw_csv = load_csv_data(uploaded_file)

# Filtramos (la función ahora tiene la 'key' interna para el slider)
df_csv_filtered, year = filter_df_by_year(df_raw_csv)

####################################################
# LAYOUT WITH TABS
####################################################

# 1. Define the tabs
tab_api, tab_csv = st.tabs(["🌍 API Data Analysis", "📈 Gapminder CSV Analysis"])

# --- TAB 1: API DATA ---
with tab_api:
    top_left, top_right = st.columns((2, 1))
    with top_left:
        st.subheader("📊 Key Metrics")
        plot_kpis(df_api_filtered)

    st.divider()

    mid_left, mid_right = st.columns(2)
    with mid_left:
        st.subheader("📊 Análisis de Correlación: Población vs PIB")
        plot_altair_insights(df_filtered=df_api_filtered, region_name=region)
    with mid_right:
        st.subheader("📋 Análisis de Barras: Población vs País")
        plot_altair_benchmark(df_filtered=df_api_filtered, region_name=region)

# --- TAB 2: CSV DATA ---
with tab_csv:
    if df_csv_filtered is not None:
        bottom_left, bottom_right = st.columns(2)

        with bottom_left:
            st.subheader("📉 Distribución Plot (Gapminder)")
            plot_life_exp_dist(df=df_csv_filtered, year=year)

        with bottom_right:
            st.subheader("🧭 Swarm Plot (Gapminder)")
            plot_seaborn_swarm(df=df_csv_filtered, year=year)
    else:
        st.warning(
            "⚠️ Por favor, sube un archivo CSV en la barra lateral para ver este análisis."
        )

```

### data_loader.py

```python
import pandas as pd
import requests
import streamlit as st


# ---------------------------------------------------
# FETCH AND CACHE API DATA
# ---------------------------------------------------
@st.cache_data
def get_data_api(url):
    response = requests.get(url)
    data = response.json()

    df = pd.json_normalize(data)
    df = df[["name.common", "region", "population", "area", "capital", "cca3"]].rename(
        columns={"name.common": "country"}
    )
    return df


@st.cache_data
def load_csv_data(uploaded_file):
    if uploaded_file is not None:
        df = pd.read_csv(uploaded_file)
        return df
    return None


def filter_df_by_year(df, year_col="year"):
    if df is not None and year_col in df.columns:
        y_min, y_max = int(df[year_col].min()), int(df[year_col].max())

        # Agregamos el parámetro key="slider_gapminder"
        selected_year = st.sidebar.slider(
            "Selecciona el Año",
            y_min,
            y_max,
            y_max,
            key="slider_gapminder",  # <-- Esto evita el error de duplicado
        )

        return df[df[year_col] == selected_year], selected_year
    return None, None

```

### api_kpis.py

```python
import streamlit as st


def kpi(title, icon, value):
    st.markdown(
        f"""
    <div class="kpi-card">
        <div class="kpi-title">{icon} {title}</div>
        <div class="kpi-value">{value:,}</div>
    </div>
    """,
        unsafe_allow_html=True,
    )


# ---- KPI LAYOUT FUNCTION ----
def plot_kpis(df):
    total_countries = len(df)
    total_population = df["population"].sum()
    avg_population = int(df["population"].mean())
    largest_area = df["area"].max()

    c1, c2, c3, c4 = st.columns(4)

    with c1:
        kpi("Total Countries", "🌍", total_countries)

    with c2:
        kpi("Total Population", "👥", total_population)

    with c3:
        kpi("Avg Population", "📊", avg_population)

    with c4:
        kpi("Largest Area (km²)", "🌐", largest_area)

```

### api_altair.py

```python
import altair as alt
import streamlit as st


# ---------------------------------------------------
# ALTAIR CHARTS
# ---------------------------------------------------
# Code	Type	Meaning	Example	Visual Result
# :Q	Quantitative	Continuous numbers	population, area	A standard numeric axis.
# :N	Nominal	Categories / Names	country, region	Distinct colors (Red, Blue, Green).
# :O	Ordinal	Ranked categories	rank, size_small_med_large	An ordered list.
# :T	Temporal	Dates and Times	year, timestamp	A timeline axis.
def plot_altair_insights(df_filtered, region_name):
    # 1. Calculate Density (Insight: How crowded is the land?)
    # We do this calculation inside the dataframe for the chart
    df_plot = df_filtered.copy()
    df_plot["density"] = df_plot["population"] / df_plot["area"].replace(0, 1)

    # Size = Area
    chart = (
        alt.Chart(df_plot)
        .mark_circle(opacity=0.7, stroke="white", strokeWidth=1)
        .encode(
            x=alt.X(
                "population:Q", title="Total Population", axis=alt.Axis(format="~s")
            ),  # Readable numbers like 1M, 10M
            y=alt.Y(
                "density:Q",
                title="Population Density (People per km²)",
                scale=alt.Scale(zero=False),
            ),
            size=alt.Size(
                "area:Q", title="Land Area", scale=alt.Scale(range=[100, 3000])
            ),
            color=alt.Color(
                "density:Q", scale=alt.Scale(scheme="magma"), title="Density Intensity"
            ),
            tooltip=[
                alt.Tooltip("country", title="Country"),
                alt.Tooltip("population", title="Population", format=","),
                alt.Tooltip("area", title="Area (km²)", format=","),
                alt.Tooltip("density", title="Density (per km²)", format=".2f"),
            ],
        )
        .properties(
            width="container",
            height=500,
            title=f"Insight: Population vs. Density in {region_name} (Linear Scale)",
        )
        .interactive()
    )

    st.altair_chart(chart, width="stretch")


def plot_altair_benchmark(df_filtered, region_name):
    # 1. Sort the data for a clean descending look
    df_plot = df_filtered.sort_values("population", ascending=False)

    # 2. Create the main Bars
    bars = (
        alt.Chart(df_plot)
        .mark_bar(cornerRadiusTopRight=5, cornerRadiusBottomRight=5)
        .encode(
            x=alt.X("population:Q", title="Population", axis=alt.Axis(format="~s")),
            y=alt.Y("country:N", sort="-x", title="Country"),
            color=alt.Color(
                "population:Q", scale=alt.Scale(scheme="purples"), legend=None
            ),
            tooltip=["country", "population", "area"],
        )
    )

    # 5. Layer them together (+)
    chart = (
        (bars)
        .properties(
            width="container",
            height=500,  # Dynamic height based on number of countries
            title=f"Population Benchmark: {region_name}",
        )
        .configure_axis(labelFontSize=12, titleFontSize=14)
    )

    st.altair_chart(chart, width="stretch")

```

### gapm_seaborn.py

```python
import matplotlib.pyplot as plt
import seaborn as sns
import streamlit as st


# ---------------------------------------------------
# SEABORN DISPLOT (Figure-level)
# ---------------------------------------------------
def plot_life_exp_dist(df, year):
    # Set the visual style
    sns.set_theme(style="darkgrid")

    # displot is a Figure-level function (returns a FacetGrid)
    # We use 'g' to access the underlying figure object
    g = sns.displot(
        data=df,
        x="lifeExp",
        kde=True,
        color="purple",
        bins=20,
        height=5,
        aspect=1.5,
    )

    # Add dynamic titles
    g.fig.suptitle(f"Distribución de la Esperanza de Vida ({year})", fontsize=16)
    g.set_axis_labels("Años de Vida", "Número de Países")

    # Render in Streamlit
    st.pyplot(g.fig)


# ---------------------------------------------------
# SEABORN SWARM PLOT (Axes-level)
# ---------------------------------------------------
def plot_seaborn_swarm(df, year):
    sns.set_theme(style="darkgrid")

    # We create the figure and axis manually to avoid 'no .fig attribute' errors
    fig, ax = plt.subplots(figsize=(10, 6))

    # Note: No 'g =' here to avoid "unused variable" warnings.
    # Gapminder columns: 'continent' and 'pop'
    sns.swarmplot(
        data=df,
        x="continent",
        y="pop",
        hue="continent",
        palette="bright",
        size=3,
        legend=False,
        ax=ax,
    )

    # Formatting the axes
    ax.set_title(f"Distribución por Continente - Año {year}", fontsize=16)
    ax.set_ylabel("Población (Escala Log)")
    ax.set_xlabel("Continente")
    ax.set_yscale("log")

    # Explicitly pass the figure to Streamlit
    st.pyplot(fig)

```